# 2 · Your first classifier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vitormeriat/text4gcn/blob/main/docs/tutorials/02_first_classifier.ipynb)

In tutorial 1 we built a graph of twelve sentences. Now we use the
**20 Newsgroups** benchmark (one of the datasets of Yao et al., 2019 and
Pereira & de Castro, DCAI 2022) and go all the way to test accuracy:

1. **Download** it and write the Text4GCN input files.
2. **Preprocess** it with `TextPipeline`.
3. **Build the graph** with `Frequency.tfidf()`.
4. **Train and evaluate** a GCN with `GNN.fit()`.

> **Runtime:** with the 4-category subset (~3.6k documents) this takes about 2 min on a Colab T4 GPU and about 6 min on CPU. It needs about 4 GB of RAM.

| # | Tutorial | Question it answers |
|---|---|---|
| 1 | Text as a graph | What does the graph look like, and why does it help? |
| 2 | **Your first classifier** | How do I go from raw text to test accuracy? |
| 3 | Choosing a graph builder | How should words be connected? |
| 4 | Refining the graph | Can a smaller graph do as well? |
| 5 | Bring your own corpus | How do I use my own (non-English) data? |

In [ ]:
%pip install -q text4gcn
# Release candidate from TestPyPI (dependencies still come from PyPI):
# %pip install -q -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "text4gcn==2.0.0rc2"

In [ ]:
import shutil
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

import text4gcn
from text4gcn import GNN, Builder, TextPipeline
from text4gcn.builder import Frequency
from text4gcn.datasets import twenty_newsgroups

print(f"text4gcn {text4gcn.__version__}")

## Configuration

The whole corpus, documents **and** vocabulary, becomes a single graph. The
full 20NG (18,846 posts) therefore needs a large GPU. We use the four
categories of the classic scikit-learn text tutorial. To scale up, add
categories or set `CATEGORIES = None`.

In [ ]:
NAME = "20NG4"
DATA_DIR = Path("data")
LOG_DIR = Path("logs")
SEED = 1234
CATEGORIES = ["alt.atheism", "soc.religion.christian", "comp.graphics", "sci.med"]

## 1. Download and write the input files

`twenty_newsgroups()` downloads the corpus with scikit-learn (already a
dependency) and keeps the official train/test split. It strips e-mail headers,
signatures and quoted replies, so the model learns from the message body, and
it drops posts left with fewer than 5 words. It then calls `save_corpus()`,
which writes the two files every Text4GCN dataset needs:

| File | One line per document |
|---|---|
| `<name>.txt` | the text, on a single line |
| `<name>.meta` | `<doc id>\t<train|test>\t<label>` |

`TextPipeline` never overwrites its outputs, so we start from an empty folder.

In [ ]:
shutil.rmtree(DATA_DIR, ignore_errors=True)
txt_path, meta_path = twenty_newsgroups(DATA_DIR, name=NAME, categories=CATEGORIES)

meta = pd.read_csv(meta_path, sep="\t", names=["id", "split", "label"])
meta.pivot_table(index="label", columns="split", values="id", aggfunc="count", margins=True)

In [ ]:
print(txt_path.read_text(encoding="utf-8").splitlines()[0][:400])

## 2. Preprocess

`execute()` writes three folders next to the input files:

| Folder | Content |
|---|---|
| `<name>.cleaned/` | text lower-cased and tokenized, without stop words and words seen fewer than `rare_count` times |
| `<name>.shuffled/` | documents shuffled with `seed`, the vocabulary, the train/test ids |
| `<name>.node_features/` | feature and label matrices; `validation_ratio` of the training documents is held out for validation |

`validation_ratio` is the **only** place where the validation split is defined.

In [ ]:
TextPipeline(
    dataset_name=NAME,
    dataset_path=DATA_DIR,
    rare_count=5,
    language="english",
    seed=SEED,
).execute(validation_ratio=0.10)

In [ ]:
vocab = (DATA_DIR / f"{NAME}.shuffled" / f"{NAME}.vocab").read_text(encoding="utf-8").split()
print(f"vocabulary: {len(vocab):,} words -> graph with {len(meta) + len(vocab):,} nodes")

## 3. Build the graph

The TextGCN graph has PMI word–word edges and TF-IDF document–word edges (see
tutorial 1). `GraphBuildStats` reports its size.

In [ ]:
stats = Frequency(dataset_name=NAME, dataset_path=DATA_DIR).tfidf(window_size=20)
stats

## 4. Train and evaluate

`GNN` loads the features and the graph and trains a 2-layer GCN with the
protocol of the paper: hidden size 200, dropout 0.5, learning rate 0.02, and
early stopping after 10 epochs without improvement. It uses CUDA when available. When training
stops, the weights with the best validation loss are restored and evaluated on
the test documents.

In [ ]:
result = GNN(
    dataset_name=NAME,
    dataset_path=DATA_DIR,
    log_dir=LOG_DIR,
    builder=Builder.Tfidf,      # the graph built in step 3
    seed=SEED,
).fit()

In [ ]:
print(f"test accuracy : {result.test_accuracy:.4f}")
print(f"epochs trained: {result.epochs_trained}")
print(f"checkpoint    : {result.model_path}")

Training curves are saved in `log_dir`.

The loss values stay close to ln(4) ≈ 1.39, and the validation curve looks
almost flat. This is expected: as in the reference TextGCN implementation, the
cross-entropy is averaged over **all** nodes, and the rows outside each split
have zeroed logits. Accuracy is the curve to read.

In [ ]:
display(Image(LOG_DIR / "metrics-acc.png"), Image(LOG_DIR / "metrics-loss.png"))

## What's next

The graph above has millions of edges, most of them word–word PMI pairs. Do we
need all of them?

**Next:** [3 · Choosing a graph builder](https://vitormeriat.github.io/text4gcn/tutorials/03_choosing_a_graph_builder/) builds the graph in other
ways and compares size, speed and accuracy.